# تدريب Veqic 1.0 — Gemma 4 E4B مع QLoRA عبر Unsloth

**البيئة:** Kaggle T4 x2 (مجاني، جلسة ≤ 12 ساعة) — كل شيء يعمل على السحابة، لا تنفيذ محلي.

**الخطة:** تثبيت Unsloth ← تحميل النموذج الأساسي بدقة 4-bit ← إضافة LoRA (r=16, alpha=16 على كل target_modules) ← فلترة أمثلة SFT من JSONL ← تطبيق chat_template ← التدريب (3 عصور) ← حفظ المحولات في `veqic-v1-lora`.

قبل التشغيل: **فعّل GPU T4 x2 من Settings ← Accelerator.**

## الخلية 1 — تثبيت Unsloth
تثبيت `unsloth` و`datasets` و`trl`. مكتبة Unsloth تتولى تثبيت اعتمادياتها الداخلية (بما فيها trl عند الحاجة)؛ لا نستخدم أي مكتبات خارج القائمة المسموحة.

In [ ]:
!pip install -q unsloth datasets trl
import unsloth  # فحص سريع أن التثبيت تم
print('unsloth جاهزة')

## الخلية 2 — تحميل النموذج الأساسي بـ 4-bit
⚠️ **تحذير إلزامي:** اسم النموذج `BASE_MODEL` أدناه غير مؤكد — *يجب* التحقق من الاسم الدقيق على Hugging Face قبل التشغيل (ابحث في صفحة حساب Unsloth على HF عن نسخة Unsloth-BNB-4bit من Gemma 4 E4B-Instruct وانسخ الاسم الحرفي من هناك).

`fp16=True` في الإعداد لاحقًا لأن T4 لا تدعم bf16.

In [ ]:
# ⚠️⚠️⚠️ تحذير: اسم النموذج غير متأكد — تحقق من الاسم على Hugging Face قبل التشغيل ⚠️⚠️⚠️
BASE_MODEL = 'unsloth/gemma-4-e4b-it-unsloth-bnb-4bit'

from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    dtype=torch.float16,      # T4 لا تدعم bf16
    load_in_4bit=True,        # QLoRA: النموذج الأساسي مجمّد بـ 4-bit
)
print('تم تحميل:', BASE_MODEL)

## الخلية 3 — إضافة محولات LoRA
r=16 و alpha=16، مع تطبيق LoRA على **كل** target_modules (attention + MLP) لتحقيق أداء أفضل من QLoRA القياسي.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],  # كل target_modules
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
model.print_trainable_parameters()

## الخلية 4 — فلترة أمثلة SFT من JSONL
نقرأ `DATA_PATH` (عدّل المسار إلى مكان رفع `data/veqic_seed_100.jsonl` على Kaggle)، ونحتفظ فقط بالأمثلة التي تحتوي على المدخلات والمخرجات معًا، ثم نصفّها إلى قالب الحوار لتطبيق chat_template.

In [ ]:
DATA_PATH = 'data/veqic_seed_100.jsonl'  # ← عدّل المسار بعد رفع البيانات إلى Kaggle

from datasets import load_dataset

raw = load_dataset('json', data_files=DATA_PATH, split='train')

def to_sft(example):
    return {
        'prompt': example.get('prompt') or example.get('input') or example.get('instruction'),
        'completion': example.get('completion') or example.get('output') or example.get('response'),
    }

sft = raw.map(to_sft)
# فلترة: أمثلة SFT فقط — يجب وجود كل من prompt و completion
sft = sft.filter(lambda ex: ex['prompt'] is not None and ex['prompt'].strip() != ''
                             and ex['completion'] is not None and ex['completion'].strip() != '')
print(f'عدد الأمثلة بعد الفلترة: {len(sft)} من أصل {len(raw)}')
sft[0]

## الخلية 5 — تطبيق chat_template والتهيئة للتدريب
نطبق قالب الحوار الخاص بالنموذج (chat_template) على كل مثال ونحوّله إلى token ids جاهزة لـ SFT.

In [ ]:
def format_with_template(example):
    messages = [
        {'role': 'user', 'content': example['prompt']},
        {'role': 'assistant', 'content': example['completion']},
    ]
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

train_ds = sft.map(format_with_template)
print(train_ds['text'][0][:500])

## الخلية 6 — التدريب بـ SFTConfig
3 عصور، معدل تعلم 2e-4، حجم دفعة 2 مع تجميع تدريجي ×4 (دفع فعلي 8)، fp16 لأن T4. الدفعات الصغيرة + التجميع التدريجي مطلوبة بسبب ذاكرة T4 المحدودة.

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field='text',
    max_seq_length=2048,
    args=SFTConfig(
        num_train_epochs=3,
        learning_rate=2e-4,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        fp16=True,          # T4 تدعم fp16 فقط (لا bf16)
        logging_steps=5,
        output_dir='veqic-v1-lora',
        save_strategy='epoch',
        warmup_ratio=0.03,
        lr_scheduler_type='cosine',
        optim='adamw_8bit',
        seed=3407,
        report_to='none',
    ),
)
trainer.train()

## الخلية 7 — حفظ المحولات (adapters)
نحفظ محولات LoRA فقط في `veqic-v1-lora` (وليس النموذج كاملًا)، ثم نحفظ الـ tokenizer معها. يمكن تنزيل المجلد من Kaggle مباشرة.

In [ ]:
model.save_pretrained('veqic-v1-lora')          # adapters فقط
tokenizer.save_pretrained('veqic-v1-lora')

import os
print('ملفات المحفوظة:', os.listdir('veqic-v1-lora'))
# اختبار سريع بعد التدريب:
FastLanguageModel.for_inference(model)
msgs = [{'role': 'user', 'content': 'مرحبًا، عرّف نفسك.'}]
ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda')
out = model.generate(input_ids=ids, max_new_tokens=64, temperature=0.7)
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))